# Lecture 2.3 — Writing Effective System Instructions: Dos and Don'ts

**Course:** OpenAI Agents SDK — Complete Course  
**Section:** 02 — Agents: Configuration & Behaviour

---

In this notebook we focus entirely on the `instructions` field of the `Agent` class — the system prompt. Every comparison is structured as a **before/after pair** so the impact of each practice is immediately visible in the output.

By the end of this notebook you will understand:
- Why specificity of role, scope, tone, and constraints all matter
- How to give an agent a distinct persona
- How to use explicit negative constraints to keep agents on-task
- How to encode output format expectations in the system prompt
- A practical dos-and-don'ts framework for writing `instructions`

## Cell 1 — Install the OpenAI Agents SDK

This notebook requires the `openai-agents` package, pinned here to a specific version so this notebook behaves exactly as it was built and recorded, regardless of what the SDK has moved on to since. The cell below installs it into your current session using the `-q` (quiet) flag to suppress verbose output.

If the package is already installed in this session, pip will confirm it quickly and move on — no harm done. This cell must be run before any import cells below.

In [1]:
!pip install openai-agents==0.18.3 -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 880.8/880.8 kB 13.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 142.6/142.6 kB 7.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 223.4/223.4 kB 9.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 3.5 MB/s eta 0:00:00


## Cell 2 — API Key Setup (Google Colab Secrets)

This notebook uses the **Google Colab Secrets** method to load your OpenAI API key. This keeps your key out of the notebook code and out of version control.

### How to add your secret in Colab:
1. Click the **🔑 key icon** in the left sidebar (or go to **Tools → Secrets**).
2. Click **"Add new secret"**.
3. Set the **Name** to `OPENAI_API_KEY`.
4. Paste your OpenAI API key into the **Value** field.
5. Enable the toggle next to this secret so the notebook can access it.
6. Run the cell below.

> **Running locally?** Set the environment variable in your terminal before launching Jupyter:  
> `export OPENAI_API_KEY="sk-...your-key-here..."`

In [2]:
from google.colab import userdata
import os

os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")

## Cell 3 — Imports

We import three things from the SDK:

| Import | Source | Purpose |
|--------|--------|---------|
| `Reasoning` | `openai.types.shared` | Controls the reasoning effort level for GPT-5 models. Must be imported from OpenAI's shared types, not from the agents package. |
| `Agent` | `agents` | The core class for creating agents. Every agent in this notebook is an instance of `Agent`. |
| `ModelSettings` | `agents` | A dataclass for configuring model-level parameters, including `reasoning` and `verbosity`. |
| `Runner` | `agents` | The execution engine. We use `Runner.run()` (with `await`) to invoke agents. |

We do **not** import `RunConfig`, `output_type`, or any tool classes in this lecture — those are covered in later sections.

In [3]:
from openai.types.shared import Reasoning
from agents import Agent, ModelSettings, Runner

## Cell 4 — What `instructions` Actually Are

Before we look at before/after examples, it helps to understand precisely what `instructions` does in the SDK.

### `instructions` is the system prompt

When you pass a string to `Agent(instructions=...)`, the SDK sends that string as the **system message** on every turn of the conversation. The model sees it on every call — not just the first one. Think of it as a standing brief that is always in scope.

This is fundamentally different from a user message. The system prompt sets the **persona**, **scope**, **tone**, and **constraints** of the agent. It is the primary mechanism for making an agent behave differently from a raw LLM call.

### Two forms: static string vs. callable

The `instructions` field accepts two forms:

| Form | When to use |
|------|-------------|
| `str` | Fixed instructions that do not change between runs. This is what we use throughout this lecture. |
| `Callable` | A function that generates instructions dynamically at runtime — for example, injecting a user's name or account tier. This is **Lecture 2.4**. |

When a callable is used, the SDK **enforces** that the function accepts exactly 2 parameters: `(run_context: RunContextWrapper, agent: Agent)`. If the signature is wrong, the SDK raises a `TypeError`. We will see this in action in Lecture 2.4 — for now, all our instructions are static strings.

### What happens when `instructions` is `None`?

If you omit `instructions` entirely, no system prompt is sent. The model operates without any standing brief. This is strongly discouraged for production agents — without instructions, the model's behaviour is undefined and inconsistent.

## Cell 5 — Vague vs. Specific Instructions

This is the most fundamental contrast in the lecture. We create two agents that receive the **same user message**, but with dramatically different instructions.

- **`agent_vague`** uses the classic placeholder: `"You are a helpful assistant."` This tells the model almost nothing about role, scope, tone, or escalation behaviour.
- **`agent_specific`** is scoped to a concrete product (`Acme Software`), a specific task (`troubleshoot issues`), a workflow rule (`always ask clarifying questions`), a length constraint (`no more than 3 sentences`), and an out-of-scope escalation path.

**What to look for in the output:**
- The specific agent should ask a clarifying question before suggesting a fix.
- The specific agent's response should be noticeably shorter.
- The specific agent knows it is scoped to Acme Software — it will not wander.

Both agents use `gpt-5.4-mini` with `reasoning=Reasoning(effort="none")` and `verbosity="low"` — the correct settings for this model to run efficiently in a notebook context.

In [4]:
agent_vague = Agent(
    name="Vague Agent",
    instructions="You are a helpful assistant.",
    model="gpt-5.4-mini",
    model_settings=ModelSettings(
        reasoning=Reasoning(effort="none"),
        verbosity="low",
    ),
)

agent_specific = Agent(
    name="Support Agent",
    instructions=(
        "You are a customer support agent for Acme Software. "
        "You help users troubleshoot issues with Acme's project management tool. "
        "Always ask clarifying questions before suggesting a fix. "
        "Keep responses concise — no more than 3 sentences. "
        "If the issue is not related to Acme Software, "
        "politely decline and redirect the user to the right resource."
    ),
    model="gpt-5.4-mini",
    model_settings=ModelSettings(
        reasoning=Reasoning(effort="none"),
        verbosity="low",
    ),
)

prompt = "My dashboard isn't loading."

result_vague = await Runner.run(agent_vague, prompt)
result_specific = await Runner.run(agent_specific, prompt)

print("Vague:\n", result_vague.final_output)
print("\nSpecific:\n", result_specific.final_output)

Vague:
 Sorry — can you tell me:

1. What dashboard/app this is?
2. What you see instead of it loading (blank page, spinner, error message)?
3. When it started happening?
4. Any recent changes (browser update, password change, network/VPN, app update)?

If you want, I can also help you troubleshoot step by step right now.

Specific:
 Can you tell me whether you see a blank page, an error message, or a spinner when the dashboard tries to load? Also, is this happening in Acme’s web app or mobile app, and did it start after a recent change?


## Cell 6 — Giving the Agent a Persona

A persona does more than set tone — it encodes **workflow behaviours** into the instructions. Notice that `agent_persona` is told to:
1. Use a specific name (`Professor Ada`)
2. Apply a teaching style (`relatable everyday analogies`)
3. Always end with a follow-up question

That third instruction is not just a style cue — it is a **behavioural rule**. The agent must produce a specific output structure (explanation + question) regardless of the topic.

This pattern — persona + workflow rule embedded in `instructions` — is one of the most powerful levers available before tools and guardrails are introduced.

**What to look for:**
- `agent_bland` will give a correct but generic textbook answer.
- `agent_persona` will frame the answer through an analogy and close with a question.
- The difference is entirely in ~60 extra words of instructions.

In [5]:
agent_bland = Agent(
    name="Bland Agent",
    instructions="Answer questions about science.",
    model="gpt-5.4-mini",
    model_settings=ModelSettings(
        reasoning=Reasoning(effort="none"),
        verbosity="low",
    ),
)

agent_persona = Agent(
    name="Professor Agent",
    instructions=(
        "You are Professor Ada, a warm and enthusiastic science educator "
        "with 20 years of teaching experience. "
        "You explain complex concepts using relatable everyday analogies. "
        "You always end your explanation with one thought-provoking "
        "follow-up question to encourage curiosity."
    ),
    model="gpt-5.4-mini",
    model_settings=ModelSettings(
        reasoning=Reasoning(effort="none"),
        verbosity="low",
    ),
)

prompt = "What is entropy?"

result_bland = await Runner.run(agent_bland, prompt)
result_persona = await Runner.run(agent_persona, prompt)

print("Bland:\n", result_bland.final_output)
print("\nPersona:\n", result_persona.final_output)

Bland:
 Entropy is a measure of how spread out or dispersed energy or matter is, and often how many possible microscopic arrangements a system can have.

In plain terms:
- **Low entropy** = more ordered, fewer possible arrangements
- **High entropy** = more disordered, more possible arrangements

In physics, it’s closely tied to the **second law of thermodynamics**: in an isolated system, entropy tends to increase over time.

Example:
- A neat deck of cards has lower entropy.
- A shuffled deck has higher entropy.

It also appears in information theory, where entropy measures **uncertainty** or **information content**.

Persona:
 Entropy is a measure of how spread out or disordered energy is in a system.

A simple analogy: think of a tidy room. When everything is organized, there are fewer ways the room can look. If you scatter the clothes, books, and papers everywhere, there are many more possible arrangements. Entropy is like that “number of possible arrangements” measure.

In physics

## Cell 7 — Constraining Scope: What the Agent Should NOT Do

Telling a model what it **should not** do is just as important as telling it what it should do. Without negative constraints, even a well-scoped agent can drift when the user sends an off-topic message.

Here we test both agents with a clearly out-of-scope prompt: a Python scripting question sent to a cooking assistant.

**Key design choices in `agent_constrained`:**
- The scope is doubly bounded: `Italian cuisine` only (a subdomain, not all cooking).
- The out-of-scope response is **fully scripted** — the agent knows exactly what to say.
- The scripted response redirects rather than just refusing, keeping the interaction positive.

**What to look for:**
- `agent_no_constraints` will likely answer the Python question, because "cooking assistant" does not explicitly exclude programming.
- `agent_constrained` will return the scripted redirect verbatim (or very close to it).

> **Note:** This approach works well before guardrails are in place. In Section 5, we introduce input guardrails that handle scope enforcement at the runner level, independently of the agent's instructions.

In [6]:
agent_no_constraints = Agent(
    name="Unconstrained Agent",
    instructions="You are a cooking assistant.",
    model="gpt-5.4-mini",
    model_settings=ModelSettings(
        reasoning=Reasoning(effort="none"),
        verbosity="low",
    ),
)

agent_constrained = Agent(
    name="Constrained Cooking Agent",
    instructions=(
        "You are a cooking assistant specialising in Italian cuisine. "
        "You only answer questions about Italian recipes, ingredients, "
        "and cooking techniques. "
        "If the user asks about anything outside Italian cooking, respond with: "
        "'I specialise in Italian cuisine — I can help you with Italian recipes "
        "and techniques. Is there something Italian I can help you with?'"
    ),
    model="gpt-5.4-mini",
    model_settings=ModelSettings(
        reasoning=Reasoning(effort="none"),
        verbosity="low",
    ),
)

prompt = "Can you help me write a Python script?"

result_unconstrained = await Runner.run(agent_no_constraints, prompt)
result_constrained = await Runner.run(agent_constrained, prompt)

print("Unconstrained:\n", result_unconstrained.final_output)
print("\nConstrained:\n", result_constrained.final_output)

Unconstrained:
 Absolutely — what do you want the script to do?

Constrained:
 I specialise in Italian cuisine — I can help you with Italian recipes and techniques. Is there something Italian I can help you with?


## Cell 8 — Output Format Instructions

When the **consumer of the output is a human**, format instructions in the system prompt are a practical and reliable approach. We can specify exact structural templates — labels, sections, and field names — and the model will follow them consistently.

**What `agent_formatted` does differently:**
- It specifies an exact template with four named fields: `HEADLINE`, `SUMMARY`, `KEY FACTS`, and `SENTIMENT`.
- The model fills in the template for any article it receives.

**An important distinction:**
When the **consumer is code** (e.g., a downstream function or API that needs to parse the response), format instructions alone are fragile — the model may vary field names or spacing. In those cases, use `output_type` with a Pydantic model to get structured, validated output. That is covered in **Lecture 2.5**.

| Output consumer | Approach |
|-----------------|----------|
| Human reader | Format instructions in `instructions` field |
| Code parser | `output_type` + Pydantic (Lecture 2.5) |

In [7]:
agent_unformatted = Agent(
    name="Unformatted Agent",
    instructions="You summarise news articles.",
    model="gpt-5.4-mini",
    model_settings=ModelSettings(
        reasoning=Reasoning(effort="none"),
        verbosity="low",
    ),
)

agent_formatted = Agent(
    name="Formatted Agent",
    instructions=(
        "You summarise news articles. "
        "Always respond in this exact format:\n"
        "HEADLINE: [one sentence]\n"
        "SUMMARY: [2-3 sentences]\n"
        "KEY FACTS: [3 bullet points]\n"
        "SENTIMENT: [Positive / Negative / Neutral]"
    ),
    model="gpt-5.4-mini",
    model_settings=ModelSettings(
        reasoning=Reasoning(effort="none"),
        verbosity="low",
    ),
)

article = (
    "OpenAI has announced a new series of models optimised for agentic workflows, "
    "featuring improved tool use and lower latency for multi-step tasks."
)

result_unformatted = await Runner.run(agent_unformatted, article)
result_formatted = await Runner.run(agent_formatted, article)

print("Unformatted:\n", result_unformatted.final_output)
print("\nFormatted:\n", result_formatted.final_output)

Unformatted:
 OpenAI announced new models optimized for agentic workflows, with better tool use and lower latency for multi-step tasks.

Formatted:
 HEADLINE: OpenAI launches new models designed for agentic workflows with faster, better tool use.
SUMMARY: OpenAI has announced a new set of models built to improve agentic workflows, especially multi-step tasks that rely on tools. The company says the models offer stronger tool use and lower latency, aiming to make automated task completion more efficient.
KEY FACTS: 
- New OpenAI models are optimized for agentic workflows.
- The models are designed for improved tool use.
- They also aim to reduce latency in multi-step tasks.
SENTIMENT: Positive


## Cell 9 — Instructions Dos and Don'ts

The four before/after comparisons above demonstrate a consistent pattern. Here is a consolidated reference:

| ✅ DO | ❌ DON'T |
|-------|----------|
| State the agent's role explicitly | Use vague openers like `"You are a helpful assistant"` alone |
| Define scope — what it does **and** what it doesn't | Leave scope open-ended and rely on the model to infer |
| Give a specific persona and tone | Leave tone unspecified for customer-facing agents |
| Include output format when the consumer is human | Mix format instructions with Pydantic when the consumer is code |
| Use explicit fallback language for out-of-scope requests | Hope the model handles off-topic gracefully |
| One agent, one job | Overload one agent with multiple unrelated responsibilities |

These principles apply to every agent you build. The underlying idea is simple: **instructions are your contract with the model**. Write them like you are briefing a new employee — role, responsibilities, constraints, and tone. The model will follow a clear brief far more reliably than it will infer one.

## Cell 10 — A Note on Instructions Length

There is no hard limit on the length of `instructions` in the SDK. However, length has real consequences:

### Token cost
Every call to an agent includes the full `instructions` string in the model's context window. A 1,000-word system prompt adds roughly 1,300 tokens to **every turn**, which compounds significantly in multi-turn or high-volume workflows.

### Focus degradation
Very long instructions can cause the model to lose focus — later instructions may be underweighted relative to earlier ones. This is an active area of research in prompt engineering.

### Practical guideline

| Instructions length | Consider... |
|--------------------|-------------|
| < 200 words | Usually fine for a single-purpose agent |
| 200–500 words | Reasonable for complex scope + tone + format requirements |
| > 500 words | Ask: is this agent doing too many jobs? |

If your system prompt exceeds ~500 words, consider whether the agent is trying to do several unrelated things at once. The right solution is usually to **split** the agent into specialist agents, each with a focused brief.

This is exactly what **Section 5** (Multi-Agent Orchestration) solves — giving each specialist agent a tight, purpose-built set of instructions and letting an orchestrator route between them.